# Gradient-Based Learning From First Principles

## Objectives

- Derive linear regression and its gradients.
- Implement gradient descent using NumPy.
- Understand forward propagation and backpropagation.
- Build a neural network with one hidden layer.
- Experiment with learning rates and validate gradients numerically.
- Benchmark our implementation against established ML libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=6, suppress=True)

### Data Generation

We create a synthetic dataset with a known linear relationship $y = wx + b + \varepsilon$, where $w = 8$, $b = 30$, and $\varepsilon$ is Gaussian noise. The input $X$ is standardised (zero mean, unit variance) to keep gradient magnitudes manageable and help gradient descent converge smoothly.

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

N = 1000
w = 8.0
b = 30.0

X = rng.uniform(0,600,size=N)
# Normalize X to prevent blowing up
X = StandardScaler().fit_transform(X.reshape(-1,1)).reshape(-1)

noise = rng.normal(0,0.6,size=N)
y = w*X + b + noise

y_pred = w * X + b
print("Residuals:", sum(y_pred - y))

### Baseline Loss

Before training, we compute the MSE between the true targets and a naïve prediction (using the initial, untrained parameters). This gives us a reference point: any useful model should reduce the loss well below this baseline.

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

loss = mse(y, y_pred)
print("MSE:", loss)

## Part A: Linear Regression from First Principles

### Gradient Derivation

For a linear model $\hat{y} = wx + b$, the mean-squared-error loss over $N$ samples is:

$$L = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

Taking partial derivatives with respect to each parameter:

$$\frac{\partial L}{\partial w} = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)\, x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)$$

At each step we update the parameters in the direction of steepest descent:

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

### Implementation

We implement batch gradient descent using NumPy. Each epoch uses the entire training dataset to calculate the gradients, then updates the parameters once.

In [ ]:
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = 0.0
        self.b = 0.0
        self.loss_history = []

    def predict(self, X):
        return X*self.w + self.b

    def fit(self, X, y):
        X = X.ravel()
        y = y.ravel()
        
        n = len(X)

        for epoch in range(self.epochs):
            if epoch%100==0:
                print(f"Epoch number {epoch} running. Results so far w = {self.w} b = {self.b}")
            
            # TODO 1: calculate predictions
            y_pred = self.predict(X)
            # TODO 2: calculate residuals
            e = y_pred-y
            # TODO 3: calculate MSE
            loss = np.mean(e**2)
            
            # TODO 4: calculate dw and db
            dl_dw  = 2/n * np.dot(X,e)
            dl_db = 2/n * np.sum(e)
            # TODO 5: update w and b
            self.w -= self.learning_rate*dl_dw
            self.b -= self.learning_rate*dl_db
            # TODO 6: record the loss
            self.loss_history.append(loss)
            pass

        return self

In [ ]:
model = LinearRegressionGD(learning_rate=0.1,epochs=30)
model.fit(X, y)

print("Learned weight:", model.w)
print("Learned bias:", model.b)
print("Final MSE:", model.loss_history[-1])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model.loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Linear Regression: Training Loss")
plt.grid(True)
plt.show()

### Verification with the Exact Solution

To confirm that gradient descent converged to the correct minimum, we solve for the optimal $w$ and $b$ analytically using the ordinary-least-squares (OLS) closed-form solution via `np.linalg.lstsq`. If our learned parameters are close to the OLS solution, gradient descent is working as expected.

In [ ]:
X_matrix = np.vstack([X, np.ones(len(X))]).T
w_opt, b_opt = np.linalg.lstsq(X_matrix, y, rcond=None)[0]

print(f"True Generative Parameters: w = {w}, b = {b}")
print(f"Exact Loss Minima (OLS):    w = {w_opt:.4f}, b = {b_opt:.4f}")

# Part B: Building Neural Networks from First Principles

Linear regression can only model linear relationships. To learn the nonlinear function $y = x^2$, we need a model with nonlinear capacity. In this section we build a one-hidden-layer neural network with a ReLU activation function, implement forward propagation and backpropagation by hand, and train it using gradient descent — all in pure NumPy.

We start with the simplest possible dataset — 1-D input, 1-D output — so that every matrix shape and gradient can be inspected directly.

In [ ]:
rng = np.random.default_rng(42)
N = 1000

X = rng.uniform(-2, 2, size=(N, 1))
noise = rng.normal(0, 0.1, size=(N, 1))

Y = X**2 + noise

plt.scatter(X[:, 0], Y[:, 0], s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Nonlinear Regression Dataset")
plt.show()

### Network Architecture

We define a single-hidden-layer network with architecture **1 → 4 → 1**: one input feature ($D = 1$), four hidden neurons ($H = 4$), and one output ($O = 1$). Weights are drawn from $\mathcal{N}(0, 0.5)$ and biases are initialised to zero.

In [ ]:
D = 1
H = 4
O = 1

W1 = rng.normal(0, 0.5, size=(D, H))
b1 = np.zeros((1, H))

W2 = rng.normal(0, 0.5, size=(H, O))
b2 = np.zeros((1, O))

### Forward Pass

The forward pass computes the network's prediction in two stages:

1. **Hidden layer:** Compute the pre-activation $Z_1 = X W_1 + b_1$, then apply ReLU to get the hidden activation $A_1 = \max(0, Z_1)$.
2. **Output layer:** Compute the prediction $\hat{Y} = A_1 W_2 + b_2$ (no activation — this is a regression task).

We cache the intermediate values $Z_1$ and $A_1$ because backpropagation will need them to compute gradients.

In [ ]:
def forward(X, W1, b1, W2, b2):
    # Hidden pre-activation
    Z1 = X@W1 + b1

    # Hidden activation
    A1 = np.maximum(0,Z1)

    # Output prediction
    Y_pred = A1@W2 + b2

    # Return intermediate values for backpropagation
    cache = {
        "Z1": Z1,
        "A1": A1,
    }

    return Y_pred, cache

In [ ]:
Y_pred, cache = forward(X, W1, b1, W2, b2)

print("X:", X.shape)
print("W1:", W1.shape)
print("Z1:", cache["Z1"].shape)
print("A1:", cache["A1"].shape)
print("W2:", W2.shape)
print("Y_pred:", Y_pred.shape)

assert Y_pred.shape == Y.shape
assert np.isfinite(Y_pred).all()

### Backward Pass (Backpropagation)

To update the weights we need $\frac{\partial L}{\partial W_1}, \frac{\partial L}{\partial b_1}, \frac{\partial L}{\partial W_2}, \frac{\partial L}{\partial b_2}$. We obtain these by applying the **chain rule** from the loss back through every layer.

Starting from the output:

$$\delta_2 = \frac{\partial L}{\partial \hat{Y}} = \frac{2}{N}(\hat{Y} - Y)$$

**Output-layer gradients:**

$$\frac{\partial L}{\partial W_2} = A_1^T \delta_2, \qquad \frac{\partial L}{\partial b_2} = \sum \delta_2$$

**Hidden-layer gradients** (propagating through ReLU):

$$\delta_1 = (\delta_2 W_2^T) \odot \mathbb{1}[Z_1 > 0]$$

$$\frac{\partial L}{\partial W_1} = X^T \delta_1, \qquad \frac{\partial L}{\partial b_1} = \sum \delta_1$$

In [ ]:
def backward(X, Y, Y_pred, cache, W2):
    Z1 = cache["Z1"]
    A1 = cache["A1"]
    N = X.shape[0]

    # Loss -> output
    delta2 = (2 / N) * (Y_pred - Y)

    # Output-layer gradients
    dW2 = A1.T @ delta2
    db2 = np.sum(delta2, axis=0, keepdims=True)

    # Output -> hidden layer
    delta_A1 = delta2 @ W2.T
    delta_Z1 = delta_A1 * (Z1 > 0)

    # Hidden-layer gradients
    dW1 = X.T @ delta_Z1
    db1 = np.sum(delta_Z1, axis=0, keepdims=True)

    return {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
    }

### Training Loop

We now combine the forward pass, loss computation, backward pass, and parameter update into a single gradient-descent training loop. Each epoch performs:

1. **Forward pass** — compute $\hat{Y}$ from the current parameters.
2. **Loss** — evaluate MSE between $\hat{Y}$ and $Y$.
3. **Backward pass** — compute gradients for every parameter.
4. **Update** — subtract $\eta \cdot \nabla$ from each parameter.

In [ ]:
# 1. Generate the dataset
rng = np.random.default_rng(42)

N = 1000
X = rng.uniform(-2, 2, size=(N, 1))
noise = rng.normal(0, 0.1, size=(N, 1))
Y = X**2 + noise

# 2. Initialize the network: 1 -> 4 -> 1
D, H, O = 1, 4, 1

W1 = rng.normal(0, 0.5, size=(D, H))
b1 = np.zeros((1, H))
W2 = rng.normal(0, 0.5, size=(H, O))
b2 = np.zeros((1, O))

# 4. Mean squared error
def mse(Y_pred, Y):
    return np.mean((Y_pred - Y) ** 2)

# Train using gradient descent
learning_rate = 0.01
epochs = 10000
loss_history = []

for epoch in range(epochs):
    # Forward pass
    Y_pred, cache = forward(X, W1, b1, W2, b2)
    loss = mse(Y_pred, Y)
    loss_history.append(loss)

    # Compute all gradients using the same parameter values
    grads = backward(X, Y, Y_pred, cache, W2)

    # Update parameters only after all gradients are computed
    W1 -= learning_rate * grads["dW1"]
    b1 -= learning_rate * grads["db1"]
    W2 -= learning_rate * grads["dW2"]
    b2 -= learning_rate * grads["db2"]

    if not np.isfinite(loss):
        raise FloatingPointError(
            f"Training diverged at epoch {epoch}. "
            "Try reducing the learning rate."
        )

    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d} | MSE: {loss:.6f}")

# Evaluate the final updated parameters
Y_pred, _ = forward(X, W1, b1, W2, b2)
final_loss = mse(Y_pred, Y)

print(f"\nInitial MSE: {loss_history[0]:.6f}")
print(f"Final MSE:   {final_loss:.6f}")


In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Training Loss")
plt.grid(True)
plt.show()

# Compare predictions against the true function
order = np.argsort(X[:, 0])

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], Y[:, 0], s=10, alpha=0.25, label="Noisy training data")
plt.plot(X[order, 0], X[order, 0]**2,
         label="True function: y = x²", linewidth=2)
plt.plot(X[order, 0], Y_pred[order, 0],
         label="Neural network", linewidth=2)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Learning a Nonlinear Function")
plt.legend()
plt.grid(True)
plt.show()


## Modular Implementation

The hand-rolled functions above work, but they hard-code a specific architecture. To experiment with different depths, widths, and activations we refactor the network into reusable building blocks:

- **`DenseLayer`** — a fully-connected layer that stores its own weights and implements `forward`, `backward`, and `update`.
- **`Activation`** — a stateless nonlinearity (ReLU or tanh) with matching `forward` and `backward` methods.
- **`MSE`** — the mean-squared-error loss, also with `forward` and `backward`.

Every component follows the same `forward` / `backward` interface, so layers can be stacked in any order and the network can be assembled declaratively.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class DenseLayer:
    def __init__(self, in_features, out_features, rng=None):
        rng = rng if rng is not None else np.random.default_rng()

        self.W = rng.normal(0, np.sqrt(2.0 / in_features), size=(in_features, out_features))
        self.b = np.zeros((1, out_features))

        self.X = None
        self.dW = None
        self.db = None

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, grad_output):
        # grad_output: dL/dZ, shape (N, out_features)
        self.dW = self.X.T @ grad_output
        self.db = np.sum(grad_output, axis=0, keepdims=True)

        # Return dL/dX for the previous layer
        grad_input = grad_output @ self.W.T
        return grad_input

    def update(self, learning_rate):
        self.W -= learning_rate * self.dW
        self.b -= learning_rate * self.db

    def compute(self,output):
        return output @ self.W + self.b


class Activation:
    def __init__(self, name="relu"):
        if name not in ("relu", "tanh"):
            raise ValueError("Activation must be 'relu' or 'tanh'.")

        self.name = name
        self.Z = None
        self.A = None

    def forward(self, Z):
        self.Z = Z

        if self.name == "relu": self.A = np.maximum(0, Z)
        else: self.A = np.tanh(Z)
            
        return self.A

    def backward(self, grad_output):
        if self.name == "relu": local_grad = (self.Z > 0).astype(float)
        else: local_grad = 1 - self.A**2

        return grad_output * local_grad

    def compute(self,output):
        if self.name == "relu": 
            return np.maximum(0, output)
        
        return np.tanh(output)


class MSE:
    def forward(self, Y_pred, Y):
        return np.mean((Y_pred - Y) ** 2)

    def backward(self, Y_pred, Y):
        # Derivative of the mean over ALL output elements
        return 2 * (Y_pred - Y) / Y.size


The `NeuralNetwork` class orchestrates the layer stack. It provides three key methods:

- **`compile`** — sets the loss function and learning rate.
- **`fit`** — runs the training loop: forward → loss → backward → update, repeated for a given number of epochs.
- **`predict`** — performs a forward pass without storing intermediate values needed for gradient computation.

In [ ]:
class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers
        self.loss_fn = MSE()
        self.learning_rate = 0.01

    def compile(self, loss=None, learning_rate=0.01):
        self.loss_fn = loss if loss is not None else MSE()
        self.learning_rate = learning_rate

    def forward(self, X):
        output = X

        for layer in self.layers:
            output = layer.forward(output)

        return output

    def backward(self, Y_pred, Y):
        grad = self.loss_fn.backward(Y_pred, Y)

        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def update(self):
        for layer in self.layers:
            if isinstance(layer, DenseLayer):
                layer.update(self.learning_rate)

    def predict(self, X):
        output = X

        for layer in self.layers:
            output = layer.compute(output)

        return output

    def fit(self, X, Y, epochs=1000, verbose=True):
        history = []

        for epoch in range(epochs):
            # Forward pass
            Y_pred = self.forward(X)
            loss = self.loss_fn.forward(Y_pred, Y)

            if not np.isfinite(loss): raise FloatingPointError(f"Training diverged at epoch {epoch}.")

            # Backward pass computes every gradient
            self.backward(Y_pred, Y)

            # Update only after all gradients are computed
            self.update()

            history.append(loss)
            if verbose and (epoch % max(1, epochs // 10) == 0):
                print(f"Epoch {epoch:5d} | MSE: {loss:.6f}")

        return history


In [ ]:
# Recreate the same dataset for a controlled comparison
rng = np.random.default_rng(42)

N = 1000
X = rng.uniform(-2, 2, size=(N, 1))
Y = X**2 + rng.normal(0, 0.1, size=(N, 1))

# Build a 1 -> 32 -> 16 -> 8 -> 1 network
model = NeuralNetwork([
    DenseLayer(1, 32, rng),
    Activation("relu"),
    DenseLayer(32, 16, rng),
    Activation("relu"),
    DenseLayer(16, 8, rng),
    Activation("tanh"),
    DenseLayer(8,1,rng)
])

model.compile(loss=MSE(), learning_rate=0.01)

history = model.fit(X, Y, epochs=5000)

# Evaluate the trained model
Y_pred = model.predict(X)
print("Final training MSE:", MSE().forward(Y_pred, Y))

# Plot loss
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training a Modular Neural Network")
plt.grid(True)
plt.show()

# Plot predictions in sorted input order
order = np.argsort(X[:, 0])

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], Y[:, 0], s=10, alpha=0.2, label="Noisy data")
plt.plot(X[order, 0], X[order, 0]**2, label="True function")
plt.plot(X[order, 0], Y_pred[order, 0], label="Network prediction")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Reproducible smoke test
rng = np.random.default_rng(42)

N = 1000
X = rng.uniform(-2, 2, size=(N, 1))
Y = X**2 + rng.normal(0, 0.1, size=(N, 1))

model = NeuralNetwork([
    DenseLayer(1, 16, rng),
    Activation("relu"),
    DenseLayer(16, 8, rng),
    Activation("relu"),
    DenseLayer(8, 1, rng),
])

model.compile(loss=MSE(), learning_rate=0.01)

# Check the forward pass before training
Y_initial = model.predict(X)

assert Y_initial.shape == Y.shape
assert np.isfinite(Y_initial).all()

initial_loss = MSE().forward(Y_initial, Y)

# Train
history = model.fit(X, Y, epochs=3000, verbose=False)

Y_final = model.predict(X)
final_loss = MSE().forward(Y_final, Y)

assert Y_final.shape == Y.shape
assert np.isfinite(Y_final).all()
assert np.isfinite(history).all()

print(f"Initial MSE: {initial_loss:.6f}")
print(f"Final MSE:   {final_loss:.6f}")
print(f"Loss reduction: {(1 - final_loss / initial_loss) * 100:.2f}%")

assert final_loss < initial_loss, (
    "Training did not improve the loss; inspect the learning rate "
    "and implementation."
)


## Experiments

With the modular implementation in place, we can now systematically explore how different hyperparameters affect learning. We vary three factors — **learning rate**, **activation function**, and **hidden-layer width** — while keeping everything else fixed so that each comparison is controlled.

In [ ]:
def make_model(seed=123):
    rng = np.random.default_rng(seed)
    return NeuralNetwork([
        DenseLayer(1, 16, rng),
        Activation("relu"),
        DenseLayer(16, 8, rng),
        Activation("relu"),
        DenseLayer(8, 1, rng),
    ])


learning_rates = [0.001, 0.01, 0.05]
results = {}

for lr in learning_rates:
    model = make_model(seed=123)
    model.compile(learning_rate=lr)

    try:
        history = model.fit(X, Y, epochs=5000, verbose=False)
        final_loss = MSE().forward(model.predict(X), Y)
        results[lr] = (history, final_loss)
        print(f"lr={lr:<6} final MSE={final_loss:.6f}")
    except FloatingPointError:
        print(f"lr={lr:<6} training diverged")

plt.figure(figsize=(9, 5))

for lr, (history, final_loss) in results.items():
    plt.plot(history, label=f"lr={lr}")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Effect of Learning Rate")
plt.legend()
plt.grid(True)
plt.show()


### Weight Initialization

How we initialise the weight matrices matters. If weights start too large, activations and gradients explode; if they start too small, signals vanish to zero and the network learns nothing.

Two principled strategies keep the variance of activations roughly constant across layers:

- **Xavier / Glorot** ($\sigma = \sqrt{2 / (n_{\text{in}} + n_{\text{out}})}$): Designed for layers with symmetric activations such as tanh or sigmoid. It balances the variance of both forward activations and backward gradients.
- **He / Kaiming** ($\sigma = \sqrt{2 / n_{\text{in}}}$): Tailored for ReLU networks, where half the activations are zeroed out. The factor of 2 compensates for this, preventing the signal from shrinking as it propagates.

Our `DenseLayer` class uses He initialization because we primarily use ReLU activations.

### Activation Functions

Activation functions introduce the nonlinearity that allows a neural network to approximate complex functions. Without them, stacking linear layers would collapse into a single linear transformation, no matter how deep the network.

- **ReLU** ($\max(0, z)$): The most widely used activation. Its gradient is 1 for positive inputs and 0 otherwise, making it computationally cheap and resistant to the vanishing-gradient problem. The downside is that neurons with negative pre-activation output a constant zero (the “dying ReLU” issue).
- **Tanh** ($\tanh(z)$): Outputs values in $(-1, 1)$, centring activations around zero. Gradients can saturate (approach zero) for large $|z|$, which slows learning in deep networks.

**Extensions not implemented here but worth noting:**
- **Leaky ReLU**: replaces the zero region of ReLU with a small slope (e.g. $0.01z$), preventing dead neurons.
- **Softmax**: converts a vector of logits into a probability distribution; used in classification output layers rather than hidden layers.

In [ ]:
def make_activation_model(activation_name, seed=123):
    rng = np.random.default_rng(seed)

    return NeuralNetwork([
        DenseLayer(1, 16, rng),
        Activation(activation_name),
        DenseLayer(16, 8, rng),
        Activation(activation_name),
        DenseLayer(8, 1, rng),
    ])


activation_results = {}

for activation_name in ["relu", "tanh"]:
    model = make_activation_model(activation_name, seed=123)
    model.compile(learning_rate=0.01)

    history = model.fit(X, Y, epochs=3000, verbose=False)
    predictions = model.predict(X)

    activation_results[activation_name] = {
        "model": model,
        "history": history,
        "predictions": predictions,
        "mse": MSE().forward(predictions, Y),
    }

    print(
        f"{activation_name:>4} | "
        f"Final MSE: {activation_results[activation_name]['mse']:.6f}"
    )

plt.figure(figsize=(9, 5))

for name, result in activation_results.items():
    plt.plot(result["history"], label=name)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("ReLU vs. Tanh: Training Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def make_width_model(width, seed=123):
    rng = np.random.default_rng(seed)

    return NeuralNetwork([
        DenseLayer(1, width, rng),
        Activation("relu"),
        DenseLayer(width, width, rng),
        Activation("relu"),
        DenseLayer(width, 1, rng),
    ])


widths = [2, 4, 8, 16, 32, 64, 128]
width_results = []

for width in widths:
    model = make_width_model(width, seed=123)
    model.compile(learning_rate=0.01)

    history = model.fit(X, Y, epochs=3000, verbose=False)
    predictions = model.predict(X)
    final_mse = MSE().forward(predictions, Y)

    width_results.append((width, final_mse))

for width, score in width_results:
    print(f"Width {width:2d} | Training MSE: {score:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(
    [w for w, _ in width_results],
    [score for _, score in width_results],
    marker="o"
)
plt.xlabel("Hidden width")
plt.ylabel("Training MSE")
plt.title("Network Capacity vs. Training Error")
plt.grid(True)
plt.show()


### Train / Validation Split

So far every model has been evaluated on its training data. To measure whether the network actually *generalises*, we create a held-out validation set that the model never sees during training. We use a 75/25 split (900 training, 300 validation) from a freshly generated dataset.

In [ ]:
rng = np.random.default_rng(2026)

X_all = rng.uniform(-2, 2, size=(1200, 1))
Y_all = X_all**2 + rng.normal(0, 0.1, size=(1200, 1))

indices = rng.permutation(len(X_all))
train_idx = indices[:900]
val_idx = indices[900:]

X_train, Y_train = X_all[train_idx], Y_all[train_idx]
X_val, Y_val = X_all[val_idx], Y_all[val_idx]

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))


In [ ]:
model = make_model()
model.fit(X_train, Y_train, epochs=5000, verbose=False)
y_pred = model.predict(X_val)
loss = MSE().forward(y_pred,Y_val)

print(loss)

## Benchmark: NumPy vs. scikit-learn

We compare our from-scratch NumPy network against scikit-learn's `MLPRegressor` on the same train/validation split. The scikit-learn model uses the **Adam** optimizer, which adapts learning rates per parameter and incorporates momentum, a significant advantage over our vanilla gradient descent. The comparison lets us verify that our implementation is correct and quantify the gap caused by the simpler optimizer.

In [ ]:
# Train our NumPy implementation
numpy_model = make_width_model(width=16, seed=123)
numpy_model.compile(learning_rate=0.01)

numpy_history = numpy_model.fit(
    X_train, Y_train,
    epochs=5000,
    verbose=False,
)

numpy_train_pred = numpy_model.predict(X_train)
numpy_val_pred = numpy_model.predict(X_val)

numpy_train_mse = MSE().forward(numpy_train_pred, Y_train)
numpy_val_mse = MSE().forward(numpy_val_pred, Y_val)

# Evaluate against the noiseless function as well
Y_val_true = X_val**2
numpy_val_function_mse = MSE().forward(numpy_val_pred, Y_val_true)

print(f"NumPy training MSE:          {numpy_train_mse:.6f}")
print(f"NumPy validation MSE:        {numpy_val_mse:.6f}")
print(f"NumPy noiseless function MSE:{numpy_val_function_mse:.6f}")


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

sklearn_model = MLPRegressor(
    hidden_layer_sizes=(16, 8),
    activation="relu",
    solver="adam",
    learning_rate_init=0.01,
    max_iter=5000,
    random_state=123,
    tol=1e-7,
)

sklearn_model.fit(X_train, Y_train.ravel())

sklearn_train_pred = sklearn_model.predict(X_train).reshape(-1, 1)
sklearn_val_pred = sklearn_model.predict(X_val).reshape(-1, 1)

sklearn_train_mse = mean_squared_error(Y_train, sklearn_train_pred)
sklearn_val_mse = mean_squared_error(Y_val, sklearn_val_pred)
sklearn_val_function_mse = mean_squared_error(
    X_val**2, sklearn_val_pred
)

print("                 NumPy       scikit-learn")
print(f"Training MSE:    {numpy_train_mse:.6f}   {sklearn_train_mse:.6f}")
print(f"Validation MSE:  {numpy_val_mse:.6f}   {sklearn_val_mse:.6f}")
print(
    f"Function MSE:    {numpy_val_function_mse:.6f}   "
    f"{sklearn_val_function_mse:.6f}"
)


In [ ]:
order = np.argsort(X_val[:, 0])

plt.figure(figsize=(9, 5))

plt.scatter(
    X_val[:, 0], Y_val[:, 0],
    s=12, alpha=0.25, label="Validation data"
)

plt.plot(
    X_val[order, 0], X_val[order, 0]**2,
    label="True function", linewidth=2
)

plt.plot(
    X_val[order, 0], numpy_val_pred[order, 0],
    label="NumPy network", linewidth=2
)

plt.plot(
    X_val[order, 0], sklearn_val_pred[order, 0],
    label="scikit-learn MLP", linewidth=2
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Validation Predictions: NumPy vs. scikit-learn")
plt.legend()
plt.grid(True)
plt.show()


## Conclusions

### What I implemented

I implemented linear regression and a feedforward neural network from first principles using NumPy. The neural network includes dense layers, activation functions, mean squared error, backpropagation, and gradient-descent parameter updates.

### What I learned

- **Learning rate:** A higher learning rate (e.g. 0.05) converges much faster in early epochs but risks overshooting and instability, while a lower rate (0.001) is stable but painfully slow. A moderate rate of 0.01 offered the best trade-off for this problem.
- **Activation functions:** ReLU networks trained faster per epoch because the gradient is either 0 or 1, avoiding the saturation problem. Tanh produced smoother predictions but converged more slowly due to gradient magnitudes shrinking in the tails.
- **Model capacity:** Increasing the hidden-layer width from 2 to ~16 neurons reduced training MSE significantly. Beyond 16–32 neurons, returns diminished quickly — the extra parameters provided little benefit on this simple quadratic target.
- **Generalization:** Training and validation MSE were very close across experiments, indicating no severe overfitting. This is expected because the target function is low-dimensional and the dataset is large relative to the model capacity.
- **Benchmark:** The scikit-learn `MLPRegressor` (using Adam) converged to a lower final loss than our vanilla SGD implementation given the same epoch budget. This is expected: Adam adapts per-parameter learning rates and uses momentum, whereas our implementation applies a single fixed learning rate to every parameter.

### Limitations

The implementation uses full-batch gradient descent and supports only a small set of layers and activation functions. The benchmark uses a synthetic regression problem and does not establish performance on more complex real-world datasets.

### Next steps

Potential extensions include mini-batch gradient descent, momentum and Adam, classification losses, and comparison with automatic differentiation in PyTorch.